In [1]:
!pip  install transformers[torch]==4.40  'medkit-lib[optional]'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 104.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 2.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 6.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 776.5/776.5 kB 7.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 435.5/435.5 kB 6.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 5.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 286.0/286.0 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 76.4 MB/s eta 0:00:0

In [8]:
from medkit.core.text import TextDocument
from medkit.text.ner.hf_entity_matcher import HFEntityMatcher
from pathlib import Path
from huggingface_hub import login
from transformers import AutoTokenizer
import tqdm
import boto3
import json
import re

In [3]:
label_map = {
    'treatment': 'DRUG',
    'problem': 'DISORDER'
}

login('HF TOKEN')

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
matcher = HFEntityMatcher(model="camila-ud/DrBERT-CASM2")
sm = boto3.client('sagemaker-runtime')

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /home/ec2-user/.cache/huggingface/token
Login successful


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/415 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [4]:
root = Path('SMM4H_2024_Task_2_test')

In [5]:
chat = [
    # train/fr_few_shot/fr_1025_lifeline_v1_FR_1971_1_1647857960.ann
   {"role": "user", "content": "From the medical report in French below, extract all the mentions of entities DRUG, DISORDER and the relationships CAUSED and TREATMENT_FOR in brat format.\nSalut <user>, pour moi, ça a commencé à l'âge de <pi>. J'ai suivi une thérapie et une cure pendant deux ans, j'ai pris des pilules et j'ai toujours eu des angoisses avant les règles. Le gynécologue m'a fait passer des tests hormonaux, qui ont révélé une pré-ménopause. J'ai pris Kliogest après avoir pris 3 hormones différentes, ça a bien marché, mais j'ai dû arrêter parce que j'avais des saignements abondants (janvier). Changement de gynécologue qui m'a mis à une dose très faible. La situation empirait de jour en jour. Par hasard, j'ai trouvé une gynécologue très sympa qui a aussi eu de gros problèmes de ménopause. J'ai l'impression que quelqu'un me comprend. Je prends maintenant Trisequens (depuis 2 mois) et Insidon pour l'anxiété et l'humeur. Mon médecin pense qu'il me faudra encore deux mois pour retrouver mon niveau d'hormones. Je te souhaite beaucoup de force. Si tu le souhaites, tu peux m'envoyer un mail. <pi> Prends soin de toi. <user>"},
   {"role": "assistant", "content": """DRUG	pilules
DISORDER	angoisses
FUNCTION	règles
FUNCTION	pré-ménopause
DRUG	Kliogest
DRUG	3 hormones différentes
DISORDER	saignements abondants
DISORDER	gros problèmes
FUNCTION	ménopause
DRUG	Trisequens
DRUG	Insidon
DISORDER	anxiété
DISORDER	humeur
FUNCTION	hormones
CAUSED Arg1:Kliogest Arg2:saignements abondants
CAUSED Arg1:pilules Arg2:angoisses
CAUSED Arg1:règles Arg2:angoisses
CAUSED Arg1:ménopause Arg2:gros problèmes
TREATMENT_FOR Arg1:Insidon Arg2:anxiété
TREATMENT_FOR Arg1:Insidon Arg2:humeur
TREATMENT_FOR Arg1:Trisequens Arg2:anxiété
TREATMENT_FOR Arg1:Trisequens Arg2:humeur
"""},
    # train/fr_few_shot/fr_1069_lifeline_v1_FR_6168_1_1648459053.ann
   {"role": "user", "content": "From the medical report in French below, extract all the mentions of entities DRUG, DISORDER and the relationships CAUSED and TREATMENT_FOR in brat format.\nBonjour, j'ai reçu de la mirtazapine pour des problèmes de sommeil. Je prends 7,5 mg le soir depuis 10 jours et je dors bien, mais j'ai une grosse somnolence durant toute la journée et j'ai très peur de prendre du poids, car je suis très mince et je ne veux pas prendre de poids. Est-ce que la mirtazapine fait toujours prendre du poids ? Jusqu'à présent, je n'ai pas remarqué d'augmentation de l'appétit, mais il paraît que le médicament diminue le métabolisme et provoque souvent de la rétention d'eau. Tout cela est censé être indépendant de la dose que tu prends. Quelqu'un en a-t-il fait l'expérience ? S'il te plaît, ne me donne pas de conseils sur ce qui est supposé aider à lutter contre les troubles du sommeil, j'ai tout essayé : Circadin (mélatonine) n'a pas fonctionné, trimpramine (faible dose), prométhazine (faible dose), les deux ont provoqué une énorme somnolence et une bouche extrêmement sèche, les antiallergiques et les antiémétiques n'aident que parfois, mais aussi avec une somnolence, le L-tryptophane sans aucun effet, le zolpidem n'agit plus que pendant 2 heures, ce qui est dommage, car je n' ai jamais été somnolente. Qui peut dire quelque chose sur la prise de 7,5 mg de mirtazapine ? Merci !"},
    {"role": "assistant", "content": """DRUG	mirtazapine
DISORDER	problèmes de sommeil
FUNCTION	dors
DISORDER	grosse somnolence
DRUG	mirtazapine
DISORDER	prendre du poids
DISORDER	augmentation de l'appétit
DRUG	médicament
DISORDER	diminue le métabolisme
DISORDER	rétention d'eau
DISORDER	troubles du sommeil
DRUG	Circadin
DRUG	mélatonine
DRUG	trimpramine
DRUG	prométhazine
DISORDER	énorme somnolence
DISORDER	extrêmement sèche
DRUG	antiallergiques
DRUG	antiémétiques
DISORDER	somnolence
DRUG	L-tryptophane
DRUG	zolpidem
DISORDER	somnolente
DRUG	mirtazapine
CAUSED Arg1:mirtazapine Arg2:augmentation de l'appétit
CAUSED Arg1:prométhazine Arg2:énorme somnolence
CAUSED Arg1:trimpramine Arg2:énorme somnolence
CAUSED Arg1:trimpramine Arg2:extrêmement sèche
CAUSED Arg1:prométhazine Arg2:extrêmement sèche
CAUSED Arg1:antiémétiques Arg2:somnolence
CAUSED Arg1:antiallergiques Arg2:somnolence
CAUSED Arg1:mirtazapine Arg2:grosse somnolence
TREATMENT_FOR Arg1:mirtazapine Arg2:problèmes de sommeil
TREATMENT_FOR Arg1:Circadin Arg2:troubles du sommeil
TREATMENT_FOR Arg1:trimpramine Arg2:troubles du sommeil
TREATMENT_FOR Arg1:prométhazine Arg2:troubles du sommeil
TREATMENT_FOR Arg1:antiallergiques Arg2:troubles du sommeil
TREATMENT_FOR Arg1:antiémétiques Arg2:troubles du sommeil
TREATMENT_FOR Arg1:L-tryptophane Arg2:troubles du sommeil
TREATMENT_FOR Arg1:zolpidem Arg2:troubles du sommeil
"""},
   {"role": "user", "content": "From the medical report in French below, extract all the mentions of entities DRUG, DISORDER and the relationships CAUSED and TREATMENT_FOR in brat format.\n{data}"},
]


In [6]:
prompt = tokenizer.apply_chat_template(chat, tokenize=False)

In [11]:
out_dir = Path('task2_fr')
errors = []
for f in tqdm.tqdm(root.iterdir()):
    if not f.name.startswith("fr_"):
        continue
    with f.open() as text:
        data = text.readlines()[0]
       
        # call llm
        response = sm.invoke_endpoint(
                EndpointName='llm-mistral-7b-v01-tgi-staging',
                Body=json.dumps({"inputs": prompt.format(data=data),
                                 "parameters": {
                                     "do_sample": False,
                                     "return_full_text": False,
                                     "max_new_tokens": 256,
                                 }}),
                ContentType="application/json",
            )
        output = response["Body"].read()
        llm_answer = json.loads(output)[0]['generated_text']
        llmlines = llm_answer.split('\n')

        # call medline drbert cams
        test_doc = TextDocument(data)
        detected_entities = matcher.run([test_doc.raw_segment])
        candidates = []
        for entity in detected_entities:
            new_label = label_map.get(entity.label, None)
            if new_label:
                candidates.append((new_label, entity.text))

        # new ne detected by llm
        for line in llmlines:
            if line.startswith("DRUG") or line.startswith("DISORDER") or line.startswith("FUNCTION"):
                try:
                    label, entity = line.split('\t')
                    for dlabel, dtext in candidates:
                        if label == dlabel and dtext == entity:
                            break
                    else:
                        candidates.append((label, entity))
                except ValueError:
                    errors.append((f.name, llmlines))

        # if they are in the text keep them
        new_candidates = []
        for label, candidate in set(candidates):
            for m in re.finditer(candidate, data):
                new_candidates.append((label, m.start(), m.end(), candidate))

        new_candidates = sorted(new_candidates, key=lambda x: x[1])
        tcount=1
        with (out_dir / (f.stem + '.ann')).open('w') as frf:
            for nc in new_candidates:
                frf.write(f"T{tcount}\t{nc[0]} {nc[1]} {nc[2]}\t{nc[3]}\n")
                tcount += 1

240it [13:09,  3.29s/it]
